In [1]:
import pandas as pd 

## Block 1: Removing columns and rows (drop)

- `df.drop(columns=["a", "b"])` removes columns; `df.drop(index=[1, 3])` removes rows by their labels.
- It returns a new table; the original is unchanged, so save the result: `a = df.drop(columns=["city"])`.
- Always write `columns=` or `index=`. `df.drop("city")` looks for a row labelled "city" and gives KeyError.
- A wrong name (including a wrong capital) gives KeyError. `errors="ignore"` skips names that do not exist, without an error.
- Row labels are kept after a drop (0 and 2 stay 0 and 2); add `reset_index(drop=True)` for a fresh 0, 1.
- Dropping by a rule: `df.drop(df[df["marks"] < 80].index)` removes the rows that pass the test. Keeping the others with a filter (`df[df["marks"] >= 80]`) gives the same result and is usually shorter.
- `del df["col"]` removes a column in place, with no copy and no result to save.
- `df.pop("col")` removes a column in place and also returns it as a Series.
- `del` and `pop` change the table itself, so make a `.copy()` first if the original must stay safe.
- A single name does not need a list, but the list form is the safe habit.

In [16]:
import pandas as pd                                    # open the pandas toolbox
students = pd.read_csv("students.csv")                 # name, city, marks (4 rows)
df = students.copy()                                   # a copy to work on

a = df.drop(columns=["city"])                          # remove one column (a list of names, saved in a)
print(list(a.columns))                                 # ['name', 'marks']
print(list(df.columns))                                # ['name', 'city', 'marks'] -> df is unchanged

print(list(df.drop(columns=["city", "marks"]).columns))    # several columns at once: ['name']

b = df.drop(index=[1, 3])                              # remove rows by their LABELS (Vivek and Meena)
print(b)                                               # Asha (0) and Ravi (2) remain, labels kept
print(df.drop(index=1).shape)                          # (3, 3): one label can be given without a list

print(df.drop(df[df["marks"] < 80].index))             # drop by a rule: find the labels of the low marks, then drop them
print(df[df["marks"] >= 80])                           # the same result, written as a filter (usually simpler)

print(df.drop(index=[1, 3]).reset_index(drop=True))    # drop rows, then renumber 0, 1

c = df.copy()
del c["city"]                                          # remove a column in place with del
print(list(c.columns))                                 # ['name', 'marks']

d = df.copy()
m = d.pop("marks")                                     # remove a column and keep it as a Series
print(m.tolist(), list(d.columns))                     # [82, 91, 67, 75] ['name', 'city']

['name', 'marks']
['name', 'city', 'marks']
['name']
   name    city  marks
0  Asha  Indore     82
2  Ravi   Delhi     67
(3, 3)
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
   name    city  marks
0  Asha  Indore     82
1  Ravi   Delhi     67
['name', 'marks']
[82, 91, 67, 76] ['name', 'city']


### Key points: Adding a row

- `df.loc[new_label] = [values]` adds a row at the bottom. It changes `df` itself.
- The list needs one value per column, in column order. A wrong count gives ValueError.
- `df.loc[len(df)] = [...]` uses the next free label, so you don't have to count.
- If the label already exists, `loc` replaces that row and adds nothing.
- `pd.concat([df, new_row_table])` stacks the tables and returns a new one. Save the result.
- After `concat` the new row keeps its own label, so a label can appear twice. Add `ignore_index=True` to renumber 0, 1, 2 ...
- The one-row table is built with each value inside a list: `pd.DataFrame({"name": ["Kiran"], ...})`.
- `df.append(...)` was removed in pandas 2. Use `concat`.
- Adding rows one by one in a loop is slow. Collect them in a list, build one table, then `concat` once.

In [21]:
df = students.copy()                                   # a copy to work on

a = df.copy()                                          # a fresh copy for this example
a.loc[4] = ["Kiran", "Pune", 88]                       # label 4 does not exist yet, so pandas makes a new row
print(a)                                               # 5 rows, Kiran at the bottom

b = df.copy()
b.loc[len(b)] = ["Kiran", "Pune", 88]                  # len(b) = 4 = the next free label, so no need to count by hand
print(b.shape)                                         # (5, 3)

c = df.copy()
c.loc[1] = ["Zed", "Goa", 50]                          # label 1 already exists, so this REPLACES Vivek's row
print(c)                                               # Vivek is gone, Zed is there, still 4 rows

new = pd.DataFrame({"name": ["Kiran"], "city": ["Pune"], "marks": [88]})   # the new row as a 1-row table
d = pd.concat([df, new])                               # stack df and new, top to bottom, into a NEW table
print(d)                                               # Kiran keeps his own label 0, so label 0 appears twice
print(pd.concat([df, new], ignore_index=True))         # ignore_index=True renumbers 0 to 4
print(df.shape)                                        # (4, 3): df itself is unchanged

    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
2   Ravi   Delhi     67
3  Meena  Indore     76
4  Kiran    Pune     88
(5, 3)
    name    city  marks
0   Asha  Indore     82
1    Zed     Goa     50
2   Ravi   Delhi     67
3  Meena  Indore     76
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
2   Ravi   Delhi     67
3  Meena  Indore     76
0  Kiran    Pune     88
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
2   Ravi   Delhi     67
3  Meena  Indore     76
4  Kiran    Pune     88
(4, 3)


### Key points: concat top to bottom

- `pd.concat([a, b])` stacks b under a and returns a new table. Save the result.
- The tables go inside one list. `pd.concat(a, b)` gives TypeError.
- Without `ignore_index=True` each table keeps its own labels, so a label can appear twice. Add `ignore_index=True` to renumber 0, 1, 2 ...
- Columns are matched by name, not by position, so a different column order still lines up.
- A column that exists in only one table is kept, and the other rows get NaN.
- A column that gains NaN becomes float64, for example int `age` turns into 20.0.
- `join="inner"` keeps only the columns that every table has, and nothing becomes NaN.
- `keys=["sec_A", "sec_B"]` records which table each row came from. Use `.loc["sec_B"]` to pull one back out.
- Use `concat` for tables with the same kind of columns, such as monthly files joined into one year.
- Stacking rows is `concat`. Matching rows by a shared column is `merge`, which comes later.

In [25]:
a = pd.DataFrame({"name": ["Asha", "Vivek"], "city": ["Indore", "Pune"], "marks": [82, 91]})    # section A
b = pd.DataFrame({"name": ["Ravi", "Meena"], "city": ["Delhi", "Indore"], "marks": [67, 75]})   # section B

print(pd.concat([a, b]))                          # stack b under a; the tables go inside ONE list
r = pd.concat([a, b], ignore_index=True)          # renumber the labels 0, 1, 2, 3
print(r)
print(r.shape)                                    # (4, 3)

c = pd.DataFrame({"name": ["Kiran"], "marks": [88], "age": [20]})   # no city column, and an extra age column
m = pd.concat([a, c], ignore_index=True)          # columns are matched by NAME, not by position
print(m)                                          # the missing boxes become NaN
print(m.dtypes.tolist())                          # age is float64 because NaN is a decimal

print(pd.concat([a, c], join="inner", ignore_index=True))   # keep only the columns both tables have

d = pd.DataFrame({"marks": [88], "name": ["Kiran"], "city": ["Pune"]})   # same columns, different order
print(pd.concat([a, d], ignore_index=True))       # still lines up correctly, because names are matched

k = pd.concat([a, b], keys=["sec_A", "sec_B"])    # keys= adds a label saying which table each row came from
print(k.loc["sec_B"])                             # pick one table's rows back out


    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
0   Ravi   Delhi     67
1  Meena  Indore     75
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
2   Ravi   Delhi     67
3  Meena  Indore     75
(4, 3)
    name    city  marks   age
0   Asha  Indore     82   NaN
1  Vivek    Pune     91   NaN
2  Kiran     NaN     88  20.0
[dtype('O'), dtype('O'), dtype('int64'), dtype('float64')]
    name  marks
0   Asha     82
1  Vivek     91
2  Kiran     88
    name    city  marks
0   Asha  Indore     82
1  Vivek    Pune     91
2  Kiran    Pune     88
    name    city  marks
0   Ravi   Delhi     67
1  Meena  Indore     75


### Key points: concat side by side

- `pd.concat([a, b], axis=1)` puts the tables next to each other. `axis=0` (the default) stacks them top to bottom.
- Rows are matched by label, not by position.
- A label found in only one table gives NaN in the other table's columns, and an int column with NaN becomes float64.
- `join="inner"` keeps only the labels present in every table.
- If the same rows are in a different order but their labels still match, the result is still correct.
- If the labels do not describe the same rows, such as after shuffling or filtering, the data gets mixed up silently. Run `reset_index(drop=True)` on the tables first when they are in the same order but have different labels.
- A Series can be glued in. Its name becomes the column name, and with no name the column is called 0.
- To add one new column from a list, use `df["new"] = [values]`. The list length must equal the row count.
- Matching rows by a shared column such as an id, instead of by label, is `merge`, which comes later.

In [ ]:
n = pd.DataFrame({"name": ["Asha", "Vivek", "Ravi"]})            # one column of names
c = pd.DataFrame({"city": ["Indore", "Pune", "Delhi"]})          # one column of cities

print(pd.concat([n, c], axis=1))                  # axis=1 means side by side (axis=0, the default, is top to bottom)

y = pd.DataFrame({"marks": [91, 82]}, index=[1, 0])   # only 2 rows, and the labels are in a different order
print(pd.concat([n, y], axis=1))                  # rows are matched by LABEL: Asha(0) gets 82, Vivek(1) gets 91, Ravi gets NaN

z = pd.DataFrame({"marks": [82, 91]})             # labels 0 and 1 only
print(pd.concat([n, z], axis=1, join="inner"))    # inner keeps only the labels present in both tables

n2 = n.copy()
n2["marks"] = [82, 91, 67]                        # a plain list of the right length is simpler for one new column
print(n2)

s = pd.Series([82, 91, 67], name="marks")         # a Series can be glued too
print(pd.concat([n, s], axis=1))                  # the Series name becomes the column name

f = n.iloc[[2, 0, 1]]                             # same students, shuffled: labels are now 2, 0, 1
g = pd.DataFrame({"marks": [67, 82, 91]})         # these marks are in the shuffled ORDER (Ravi, Asha, Vivek)
print(pd.concat([f, g], axis=1))                  # TRAP: matched by label, so Ravi gets 91 (wrong)
print(pd.concat([f.reset_index(drop=True), g], axis=1))   # reset the labels first, then the order matches

    name    city
0   Asha  Indore
1  Vivek    Pune
2   Ravi   Delhi
    name  marks
0   Asha   82.0
1  Vivek   91.0
2   Ravi    NaN
    name  marks
0   Asha     82
1  Vivek     91
    name  marks
0   Asha     87
1  Vivek     65
2   Ravi     99
    name  marks
0   Asha     82
1  Vivek     91
2   Ravi     67


### Key points: insert and reordering columns

- `df["new"] = values` adds a column at the far right.
- `df.insert(position, "name", values)` adds a column at a chosen position. Position 0 is the front.
- `insert` changes the table itself and returns nothing, so don't write `df = df.insert(...)`.
- `insert` with a name that already exists gives ValueError. `allow_duplicates=True` forces it, but duplicate names are best avoided.
- The position must be between 0 and the number of columns. A bigger number gives IndexError.
- The values can be a list of the right length, one single value (repeated for every row), or a column calculated from the table.
- To reorder columns, select them in the order you want: `df[["marks", "name", "city"]]`. This returns a new table.
- To move one column to the front without typing every name: `df[["city"] + [c for c in df.columns if c != "city"]]`.
- Reordering only changes the display order. The data and the labels stay the same.
- A wrong column name in the reorder list gives KeyError. Check `list(df.columns)` first.


In [41]:
df = students.copy()

a = df.copy()
a["grade"] = ["A", "A", "B", "B"]                          # a new column always goes at the far right
print(list(a.columns)) 

b = df.copy()
b.insert(1,"roll",[101,102,103,104])
print(b)

e = df.copy()
e.insert(0,"rank", e["marks"].rank(ascending=False).astype(int))
print(e)

print(list(df[["marks", "name", "city"]].columns)) 

['name', 'city', 'marks', 'grade']
    name  roll    city  marks
0   Asha   101  Indore     82
1  Vivek   102    Pune     91
2   Ravi   103   Delhi     67
3  Meena   104  Indore     76
   rank   name    city  marks
0     2   Asha  Indore     82
1     1  Vivek    Pune     91
2     4   Ravi   Delhi     67
3     3  Meena  Indore     76
['marks', 'name', 'city']
